# 01 - Check downloads
Confirms every file is where the notebooks expect it. Run this first and send the output if anything says MISSING.

In [7]:
import sys
print(sys.executable)
import pandas, pyarrow, sklearn, matplotlib, bs4
print("all packages found")

c:\Users\yassi\AppData\Local\Python\pythoncore-3.14-64\python.exe
all packages found


In [8]:
import sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT / "src"))
from paths import *
print("repo root:", ROOT)

repo root: C:\Users\yassi\OneDrive\Documents\GitHub\WhoWroteIt-


In [9]:
import pandas as pd

def status(label, ok, extra=""):
    print(f"{'OK     ' if ok else 'MISSING'} {label} {extra}")

html_dir = find_dir(CODENET_DIR, "problem_descriptions")
n_html = len(list(html_dir.rglob("*.html"))) if html_dir else 0
status("CodeNet problem_descriptions", html_dir is not None, f"-> {html_dir} ({n_html} html files)")

plist = find_file(CODENET_DIR, "problem_list.csv")
status("CodeNet problem_list.csv", plist is not None, f"-> {plist}")
if plist:
    display(pd.read_csv(plist).head(3))

emb_csv = find_embeddings_csv()
status("AIGCodeSet embeddings csv", emb_csv is not None,
       f"-> {emb_csv} ({emb_csv.stat().st_size/1e6:.0f} MB)" if emb_csv else "")
if emb_csv:
    cols = pd.read_csv(emb_csv, nrows=2).columns.tolist()
    print("   embeddings csv columns:", cols)

OK      CodeNet problem_descriptions -> C:\Users\yassi\OneDrive\Documents\GitHub\WhoWroteIt-\data\external\codenet\Project_CodeNet\problem_descriptions (3999 html files)
OK      CodeNet problem_list.csv -> C:\Users\yassi\OneDrive\Documents\GitHub\WhoWroteIt-\data\external\codenet\Project_CodeNet\metadata\problem_list.csv


,id,name,dataset,time_limit,memory_limit,rating,tags,complexity
0,p00000,QQ,AIZU,1000.0,131072.0,NaN,NaN,NaN
1,p00001,List of Top 3 Hills,AIZU,1000.0,131072.0,NaN,NaN,NaN
2,p00002,Digit Number,AIZU,1000.0,131072.0,NaN,NaN,NaN


OK      AIGCodeSet embeddings csv -> C:\Users\yassi\OneDrive\Documents\GitHub\WhoWroteIt-\data\external\aigcodeset\all_data_with_ada_embeddings_will_be_splitted_into_train_test_set.csv (265 MB)
   embeddings csv columns: ['problem_id', 'submission_id', 'status_in_folder', 'LLM', 'code', 'ada_embedding', 'label', 'lines', 'code_lines', 'comments', 'functions', 'blank_lines']


In [10]:
for s in ["train", "val", "test"]:
    p = PROCESSED / f"{s}.parquet"
    status(f"split {s}", p.exists())
    if p.exists():
        df = pd.read_parquet(p)
        print("   rows:", len(df))
        print("   columns:", df.columns.tolist())

p = INTERIM / "problems.parquet"
status("data/interim/problems.parquet", p.exists())
if p.exists():
    pr = pd.read_parquet(p)
    print("   rows:", len(pr), "| columns:", pr.columns.tolist())
    display(pr.head(2))

OK      split train 
   rows: 4670
   columns: ['sample_id', 'source', 'submission_id', 'problem_id', 'language', 'code_raw', 'code_norm', 'label', 'llm_name', 'scenario', 'status', 'n_lines', 'n_chars', 'code_hash', 'parses_ok', 'role', 'split', 'in_detector', 'rag_eligible']
OK      split val 
   rows: 1005
   columns: ['sample_id', 'source', 'submission_id', 'problem_id', 'language', 'code_raw', 'code_norm', 'label', 'llm_name', 'scenario', 'status', 'n_lines', 'n_chars', 'code_hash', 'parses_ok', 'role', 'split', 'in_detector', 'rag_eligible']
OK      split test 
   rows: 1009
   columns: ['sample_id', 'source', 'submission_id', 'problem_id', 'language', 'code_raw', 'code_norm', 'label', 'llm_name', 'scenario', 'status', 'n_lines', 'n_chars', 'code_hash', 'parses_ok', 'role', 'split', 'in_detector', 'rag_eligible']
OK      data/interim/problems.parquet 
   rows: 800 | columns: ['problem_id', 'title', 'description', 'n_chars', 'has_description', 'in_aigcodeset_problem']


,problem_id,title,description,n_chars,has_description,in_aigcodeset_problem
0,p00000,p00000,,0,False,False
1,p00001,p00001,,0,False,False


In [11]:
pred_files = sorted(RESULTS.rglob("preds_*.parquet"))
status("prediction files from Nour", len(pred_files) > 0, f"({len(pred_files)} found)")
for f in pred_files:
    d = pd.read_parquet(f)
    print("  ", f.relative_to(ROOT), "| rows:", len(d), "| columns:", d.columns.tolist())

for f in ["features_train.parquet", "features_val.parquet", "features_test.parquet"]:
    status(f"features/{f}", (FEATURES / f).exists())

OK      prediction files from Nour (9 found)
   results\preds_combined_lr.parquet | rows: 2014 | columns: ['sample_id', 'split', 'y_true', 'y_score', 'y_pred']
   results\preds_feat_lr.parquet | rows: 2014 | columns: ['sample_id', 'split', 'y_true', 'y_score', 'y_pred']
   results\preds_feat_rf.parquet | rows: 2014 | columns: ['sample_id', 'split', 'y_true', 'y_score', 'y_pred']
   results\preds_feat_xgb.parquet | rows: 2014 | columns: ['sample_id', 'split', 'y_true', 'y_score', 'y_pred']
   results\preds_feat_xgb_with_trailing_ws.parquet | rows: 2014 | columns: ['sample_id', 'split', 'y_true', 'y_score', 'y_pred']
   results\preds_tfidf_char_lr.parquet | rows: 2014 | columns: ['sample_id', 'split', 'y_true', 'y_score', 'y_pred']
   results\preds_tfidf_char_svm.parquet | rows: 2014 | columns: ['sample_id', 'split', 'y_true', 'y_score', 'y_pred']
   results\preds_tfidf_token_lr.parquet | rows: 2014 | columns: ['sample_id', 'split', 'y_true', 'y_score', 'y_pred']
   results\preds_tfidf_t